# Part 7: Optical Change Detection

建立双时相光学变化检测最小闭环：配准、Siamese 特征、差异融合、阈值与 F1/IoU。

**先修要求**：完成前序章节，了解 NumPy、张量 shape 与训练/验证划分。

## 学习目标与路线

1. 明确问题与输入输出
2. 理解核心结构
3. 运行最小实验
4. 检查指标和 shape
5. 完成练习与遥感迁移

`image_t1 + image_t2 → shared encoder → feature difference/fusion → change logits → binary mask`

In [ ]:
from pathlib import Path
import json
ROOT = Path.cwd().resolve()
while not (ROOT / 'scripts').exists() and ROOT.parent != ROOT: ROOT = ROOT.parent
print('repository:', ROOT)

## 核心概念

- **Co-registration**：亚像素错位也会制造伪变化，进入模型前先对齐。
- **Siamese encoder**：两个时相共享权重，便于比较一致的特征空间。
- **Change fusion**：可用绝对差、拼接、注意力或双向交互融合。
- **Metrics**：变化像素稀少，应报告 precision、recall、F1、IoU，而非只报 accuracy。

## 本地真实训练结果

长训练由脚本执行；这里读取本机已经生成的指标和产物。GitHub 中不会包含 `outputs/`。

In [ ]:
output = ROOT / r'outputs/part7_change_weighted'
metrics_path = output / 'metrics.json'
print('local output:', output)
print(json.dumps(json.loads(metrics_path.read_text(encoding='utf-8')), indent=2) if metrics_path.exists() else '尚未运行：请执行下方命令。')

## 可恢复的脚本训练

默认不在 Notebook 内启动长训练。将 `RUN_SHORT_TRAIN` 改为 `True` 可执行一次小规模 smoke run；正式命令请在终端执行。

In [ ]:
RUN_SHORT_TRAIN = False
command = r'''conda run --no-capture-output -n remote-sensing-dl python scripts/real_training/train_change.py --data-root data/levir_cd --output outputs/part7_change_weighted'''
print(command)
if RUN_SHORT_TRAIN:
    import subprocess
    subprocess.run(command, cwd=ROOT, shell=True, check=True)

## 结果解释

正式结果必须包含非空 checkpoint、完整 epoch 日志、验证指标与预测图；出现缺失时先检查数据配对、CUDA 与配置，而不要直接扩大模型。

## 练习

人为将 t2 平移 1 个像素，比较伪变化数量；再讨论真实项目的配准质量检查。

In [ ]:
# 答案脚手架
experiment = {'hypothesis': '', 'controlled_variables': [], 'metric': '', 'expected_failure': ''}
print(experiment)

## 常见坑与扩展

**常见坑**：时相未配准；季节/光照变化被当成地物变化；正负样本极不平衡；同一区域跨集合泄漏。

**扩展**：替换为真实遥感数据时，记录传感器、波段、分辨率、空间划分和随机种子。

## 完成检查

- [ ] 所有单元格从头运行
- [ ] 指标实现通过 sanity check
- [ ] 没有 train/val 空间泄漏
- [ ] 配置和随机种子已记录
- [ ] 至少分析一个失败案例